# Generates crops ad pickle file using generate_crops

This notebook crops images according to bounding box coordinates (one per sulcus)

# Imports

In [ ]:
import sys
import os
import glob
import json
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import the aims module
from soma import aims
# the brainplot package
import colorado as cld

print((sys.version))

In [ ]:
import anatomist.notebook as ana
a = ana.Anatomist()
print(a.headless_info.__dict__)


In [ ]:
# Uncomment the following line if you have not installed deep_folding with pip install -e .
# sys.path.append((os.path.abspath('../')))
import deep_folding
from deep_folding.brainvisa.utils.constants import _ALL_SUBJECTS
print(inspect.getfile(deep_folding))

# User-specific variables

In [ ]:
sulcus = 'S.T.s.ter.asc.ant.'

In [ ]:
side = 'L'

We now assign path names and other user-specific variables.

The source directory is where the database lies. It contains the morphologist analysis subfolder ANALYSIS/3T_morphologist


In [ ]:
src_dir = os.path.join(os.getcwd(), '../data/source/unsupervised')
src_dir = os.path.abspath(src_dir)
print(("src_dir = " + src_dir))

The target directory tgt_dir is where the files will be saved

In [ ]:
tgt_dir = os.path.join(os.getcwd(), '../data/target')
tgt_dir = os.path.abspath(tgt_dir)
print(("tgt_dir = " + tgt_dir))

In [ ]:
bbox_dir = os.path.join(os.getcwd(), '../data/reference/bbox')
bbox_dir = os.path.abspath(bbox_dir)
print(("bbox_dir = " + bbox_dir))

In [ ]:
mask_dir = os.path.join(os.getcwd(), '../data/reference/mask')
mask_dir = os.path.abspath(mask_dir)
print(("mask_dir = " + mask_dir))

In [ ]:
print((sys.argv))

# Illustration of main program uses

We will first use the program with no effect by using number of subjects set to 0, or by calling the help function

### Using external calls

In [ ]:
!python ../deep_folding/brainvisa/generate_crops.py -n 0 -t tgt_local_dir

In [ ]:
# Clean
!rm -rf tgt_local_dir

In [ ]:
!python ../deep_folding/brainvisa/compute_bounding_box.py --help

### By using the main function call

In [ ]:
from deep_folding.brainvisa import generate_crops
print((generate_crops.__file__))

In [ ]:
args = "-n 0 -o " + tgt_dir
argv = args.split(' ')

In [ ]:
generate_crops.main(argv)

In [ ]:
args = "--help"
argv = args.split(' ')

In [ ]:
generate_crops.main(argv)

### By using the API function call

In [ ]:
src_dir_subject = f"{src_dir}/ANALYSIS/3T_morphologist"
skeleton_raw_dir = f"{tgt_dir}/skeletons/raw"
transform_dir = f"{tgt_dir}/transforms"
skeleton_1mm_dir = f"{tgt_dir}/skeletons/1mm"
src_dir_supervised  = os.path.join(os.getcwd(), '../data/source/supervised')
bbox_dir = f"{tgt_dir}/bbox"
print(src_dir)

In [ ]:
def are_arrays_almost_equal(arr1, arr2, epsilon, max_number_different_pixels):
    """Returns True if at most max_number_different_pixels pixels of arrays arr1 and arr2 
    differ by more than epsilon
    
    """
    difference = (abs(arr1-arr2) >= epsilon)
    number_different_pixels = np.count_nonzero(difference)
    return number_different_pixels <= max_number_different_pixels, number_different_pixels

# First generate hemisphere skeletons and bounding box

In [ ]:
from deep_folding.brainvisa import generate_skeletons
from deep_folding.brainvisa import generate_ICBM2009c_transforms
from deep_folding.brainvisa import resample_files
from deep_folding.brainvisa import compute_bounding_box

In [ ]:
generate_skeletons.generate_skeletons(
    src_dir=src_dir_subject,
    skeleton_dir=skeleton_raw_dir,
    side=side,
    number_subjects=_ALL_SUBJECTS)

In [ ]:
generate_ICBM2009c_transforms.generate_ICBM2009c_transforms(
    src_dir=src_dir_subject,
    transform_dir=transform_dir,
    side=side)

In [ ]:
resample_files.resample_files(
    src_dir=skeleton_raw_dir,
    input_type='skeleton',
    resampled_dir=skeleton_1mm_dir,
    transform_dir=transform_dir,
    side=side)

In [ ]:
compute_bounding_box.compute_bounding_box(
    src_dir=src_dir_supervised,
    bbox_dir=bbox_dir,
    sulcus=sulcus,
    side=side,
    out_voxel_size=1.)


# Crops with mask and with nearest-neighbour interpolation

## Main program

In [ ]:
tgt_dir_nearest = os.path.join(os.getcwd(), '../data/target/crops/nearest')
tgt_dir_nearest = os.path.abspath(tgt_dir_nearest)
print(("tgt_dir = " + tgt_dir_nearest))

Just to warp up, with number of subjects to 0

In [ ]:
generate_crops.generate_crops(
    src_dir=skeleton_1mm_dir,
    crop_dir=tgt_dir_nearest,
    bbox_dir=bbox_dir,
    cropping_type='bbox',
    list_sulci=sulcus,
    side=side,
    number_subjects=0)

In [ ]:
skeleton_1mm_dir

In [ ]:
generate_crops.generate_crops(
    src_dir=skeleton_1mm_dir,
    crop_dir=tgt_dir_nearest,
    bbox_dir=bbox_dir,
    cropping_type="bbox",
    list_sulci=sulcus,
    side=side)

## Result analysis

### Analysis of the inputs

In [ ]:
# Gets source file as numpy array
skeleton_dir = os.path.join(src_dir, "ANALYSIS/3T_morphologist/100206/t1mri/default_acquisition/default_analysis/segmentation")
vol_source_file = glob.glob(skeleton_dir + '/' + side + '*.nii.gz')[0]
vol_source = aims.read(vol_source_file)
arr_source = vol_source.arraydata()
print("shape of source skeleton = ", arr_source.shape)

In [ ]:
np.unique(arr_source)

In [ ]:
pd.value_counts(np.resize(arr_source, arr_source.size))

In [ ]:
mask_file = glob.glob(mask_dir + '/' + side + '/*.nii.gz')[0]
vol_mask = aims.read(mask_file)
arr_mask = vol_mask.arraydata()
print("shape of mask = ", arr_mask.shape)

In [ ]:
np.unique(arr_mask)

### Analysis of the outputs

Prints the list of files of the target directory

In [ ]:
print("Files in crops target directory:")
print(tgt_dir_nearest)
print(('\n'.join(os.listdir(tgt_dir_nearest + '/' + side + 'crops'))))

In [ ]:
tgt_json_file = glob.glob(tgt_dir_nearest + '/*.json')[0]
print("tgt_json_file = ", tgt_json_file, '\n')
with open(os.path.join(tgt_dir_nearest, tgt_json_file), 'r') as f:
    data_tgt = json.load(f)
    print((json.dumps(data_tgt, sort_keys=True, indent=4)))

Obtained output (we read the cropped file from the target directory):

In [ ]:
# Gets target crop as numpy array
cropped_target_dir = os.path.join(tgt_dir_nearest, side+'crops')
vol_target_file = glob.glob(cropped_target_dir + '/' + '*.nii.gz')
vol_target = aims.read(vol_target_file[0])
arr_target = vol_target.arraydata()
print("shape of target cropped image = ", arr_target.shape)

In [ ]:
np.unique(arr_target)

The scope here is to compare the different numbers present on the target array and on the source array:

In [ ]:
np.around(pd.value_counts(np.resize(arr_target, arr_target.size))/arr_target.size*100, 1)

In [ ]:
np.around(pd.value_counts(np.resize(arr_source, arr_source.size))/arr_source.size*100, 1)

### Visualization

In [ ]:
print("Files in crops nearest target directory:")
print(('\n'.join(os.listdir(tgt_dir_nearest + '/' + side + 'crops'))))

In [ ]:
target_file_dir = tgt_dir_nearest + '/' + side + 'crops'
target_file_nearest = glob.glob(target_file_dir + "/*.nii.gz")[0]
print(target_file_nearest)

In [ ]:
# load source skeleton data (the SliceableObject)
object_anat = a.loadObject(target_file_nearest)

# create an Axial window in anatomist
w = a.createWindow('Axial')
w.addObjects(object_anat)

In [ ]:
print(vol_source_file)
# load source skeleton data (the SliceableObject)
a_source = a.loadObject(vol_source_file)
print(a_source)
# create an Axial window in anatomist
w1 = a.createWindow("Axial")
w1.addObjects(a_source)